# 3.4 Claude-Style HITL Agent

## What you will learn

- How Claude's `ask_user_input` widget pattern works under the hood
- How to replicate it exactly using `HumanInTheLoopMiddleware` with **one tool only**
- How the LLM handles the actual task itself — no action tools needed
- Why this is cleaner and more general than tool-per-usecase agents

---

### How Claude actually works

```
Claude has ONE tool:   ask_user_input(question, options)
Claude's knowledge:   handles the actual task (drafts email, gives info, writes plan)

User: 'Find me flights from India to China'
  Claude: needs departure city → calls ask_user_input()
  Widget appears → user picks 'Hyderabad'
  Claude: needs date → calls ask_user_input()
  Widget appears → user picks 'April 15'
  Claude: has everything → responds with flight options from its knowledge

User: 'Draft a follow-up email to my client'
  Claude: needs tone → calls ask_user_input()
  Widget appears → user picks 'Apologetic'
  Claude: has everything → writes the email draft directly

No search_flights tool. No send_email tool. Just the model + one HITL gate.
```

### Our LangChain agent replicates this exactly

```
interrupt_on = {'ask_user_input': True}   ← the ONLY tool. The ONLY entry.

LLM handles the task with its own knowledge.
When it needs input → calls ask_user_input → PAUSES.
Human answers → resumes → LLM continues.
When it has everything → responds directly. No tool call. No action tool.
```

In [1]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# ============================================================
# CELL 2: Tools
# ============================================================
#
# TWO categories of tools:
#
# 1. HITL tool — interrupt_on=True (middleware pauses execution)
#
#    ask_user_input:
#      The only tool the middleware watches.
#      LLM calls it when it needs user input.
#      Graph pauses → human answers → resumes → LLM continues.
#
# 2. Auto-execute tools — NOT in interrupt_on (run silently)
#
#    tavily_search:
#      LLM calls this when it needs current/external information
#      that its training knowledge doesn't have.
#      Runs immediately, result returned as ToolMessage, no pause.
#      Examples: live flight prices, today's news, latest drug approvals,
#                real-time weather, recent events, current stock prices.
#
# Decision logic the LLM applies on every turn:
#
#   Need user input?      → ask_user_input()   (pauses)
#   Need external data?   → tavily_search()    (auto-runs)
#   Have everything?      → respond directly   (no tool call)

from typing import List
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults


# ── HITL Gate — middleware watches only this ──────────────────────────────────
@tool
def ask_user_input(
    question: str,
    options: List[str],
    allow_freetext: bool = True,
    user_answer: str = ""
) -> str:
    """
    Ask the user for input needed to complete the task.
    Call once per missing piece of information — ONE question at a time.
    Generate options SPECIFIC to the user's exact request context.
    Leave user_answer empty — the human fills it in via the review UI.
    Returns the human's answer as a string.
    """
    return user_answer


# ── Auto-execute tools — not in interrupt_on, run without pausing ─────────────
tavily_search = TavilySearchResults(
    max_results=5,
    description=(
        "Search the web for current, real-time, or external information. "
        "Use when the user asks about recent events, live prices, current news, "
        "latest research, real-time data, or anything that may have changed "
        "since your training cutoff. Do NOT use for general knowledge you already know."
    )
)


ALL_TOOLS = [
    ask_user_input,   # HITL — middleware intercepts this
    tavily_search,    # auto-execute — runs silently when LLM needs web data
]

print("Tools:", [t.name for t in ALL_TOOLS])
print("interrupt_on will watch: ask_user_input only")
print("tavily_search auto-executes — no pause, no approval")

Tools: ['ask_user_input', 'tavily_search_results_json']
interrupt_on will watch: ask_user_input only
tavily_search auto-executes — no pause, no approval


/var/folders/dq/76c6b9rx7tv5fhypn7n6fx800000gn/T/ipykernel_72754/1322049632.py:31: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
/var/folders/dq/76c6b9rx7tv5fhypn7n6fx800000gn/T/ipykernel_72754/1322049632.py:53: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(


In [6]:
# ============================================================
# CELL 3: Build the Agent — Claude Style
# ============================================================
#
# interrupt_on has ONE entry: ask_user_input.
# That is the complete middleware configuration.
#
# The LLM is the action engine.
# It drafts emails, plans trips, gives pharma info, schedules meetings
# — all from its own knowledge once it has the inputs it needs.
#
# System prompt teaches the LLM three things:
#   1. Call ask_user_input for every missing piece of info
#   2. Generate specific, contextual options — never generic
#   3. Once it has everything → respond directly, no tool call needed

from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


SYSTEM_PROMPT = """You are a helpful assistant. You can help with anything the user asks.

You have two tools:
  ask_user_input  — ask the user when YOU need their input to proceed
  tavily_search   — search the web when YOU need current or external information

## When to call ask_user_input

Call it when the request is ambiguous and only the USER can resolve it:
  - A specific preference   (which city, which tone, which format)
  - A key unknown detail    (recipient name, travel date, target audience)
  - An ambiguous choice     (multiple valid interpretations)

Do NOT call it when:
  - The request has enough context to answer well  → respond directly
  - You need external data                         → use tavily_search instead
  - A reasonable default clearly applies           → use it and respond

## When to call tavily_search

Call it when you need information that may have changed since your training
or that requires real-time / external data:
  - Current news, recent events, today's prices
  - Latest research papers, recent drug approvals
  - Live flight schedules, current availability
  - Real-time weather, stock prices, sports scores
  - Any fact the user would expect to be up-to-date

Do NOT call it for general knowledge you already know well.

## How to generate options for ask_user_input

Options must be SPECIFIC to the user's exact request — never generic:
  User: 'flights from India to China'
    question: 'Which city in India are you flying from?'
    options:  ['Delhi (DEL)', 'Mumbai (BOM)', 'Hyderabad (HYD)', 'Chennai (MAA)', 'Other']

  User: 'follow-up email'
    question: 'What tone should the email have?'
    options:  ['Professional and polite', 'Apologetic', 'Firm / urgent', 'Friendly']

  NEVER use: ['Option A', 'Option B'] — always derive from context.

## After collecting inputs or searching

Respond completely and directly. Give the user something fully usable.
"""


agent = create_agent(
    model="claude-sonnet-4-6",
    tools=ALL_TOOLS,                    # ask_user_input + tavily_search
    checkpointer=InMemorySaver(),       # REQUIRED — persists state across pause/resume
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "ask_user_input": True, # ONLY this. tavily_search is not listed — auto-executes.
            },
            description_prefix="Input required",
        ),
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("Agent ready")
print("Tools:        ask_user_input, tavily_search")
print("interrupt_on: {ask_user_input: True} — tavily_search auto-executes, no pause")

Agent ready
Tools:        ask_user_input, tavily_search
interrupt_on: {ask_user_input: True} — tavily_search auto-executes, no pause


In [7]:
# ============================================================
# CELL 4: The HITL Loop
# ============================================================
#
# The simplest possible loop — one interrupt type, one handler.
#
# Interrupt payload (identical structure to notebook 3.3):
#
#   response['__interrupt__'][0].value = {
#       'action_requests': [{
#           'name': 'ask_user_input',
#           'args': {
#               'question':      'Which city in India are you departing from?',
#               'options':       ['Delhi (DEL)', 'Mumbai (BOM)', 'Hyderabad (HYD)', ...],
#               'allow_freetext': True,
#               'user_answer':   ''    ← we inject the answer here via 'edit'
#           }
#       }],
#       'review_configs': [{
#           'action_name': 'ask_user_input',
#           'allowed_decisions': ['approve', 'edit', 'reject']
#       }]
#   }
#
# Resume with 'edit' injecting user_answer.
# Tool runs → returns user_answer as ToolMessage → LLM reads it.
# Loop continues until LLM has all inputs → responds directly.

from langchain.messages import HumanMessage
from langgraph.types import Command


def handle_ask_user_input(action_request: dict) -> Command:
    """
    Display the LLM-generated question + options.
    Resume with 'edit' to inject the human's answer as user_answer.
    Identical resume format to notebook 3.3.
    """
    args           = action_request["args"]
    question       = args["question"]
    options        = args["options"]
    allow_freetext = args.get("allow_freetext", True)

    print("\n" + "─" * 56)
    print(f"  {question}\n")
    for i, opt in enumerate(options, 1):
        print(f"  {i}. {opt}")
    if allow_freetext:
        print("  (or type your own answer)")
    print("─" * 56)

    raw = input("  Your answer (number or text): ").strip()

    # Resolve number → option text
    if raw.isdigit() and 1 <= int(raw) <= len(options):
        user_answer = options[int(raw) - 1]
    else:
        user_answer = raw

    print(f"  ✓ {user_answer}")

    # Resume with edit — inject user_answer.
    # ask_user_input runs and returns user_answer as a ToolMessage to the LLM.
    return Command(
        resume={
            "decisions": [{
                "type": "edit",
                "edited_action": {
                    "name": "ask_user_input",
                    "args": {
                        "question":       question,
                        "options":        options,
                        "allow_freetext": allow_freetext,
                        "user_answer":    user_answer,   # ← injected
                    }
                }
            }]
        }
    )


def run_claude_style_agent(user_message: str, thread_id: str):
    """
    Run the Claude-style agent with HITL.

    Loop:
      __interrupt__ present  → LLM needs input → show Q&A → resume
      __interrupt__ absent   → LLM has everything → final response
    """
    config = {"configurable": {"thread_id": thread_id}}
    current_input = {"messages": [HumanMessage(content=user_message)]}

    print(f"\n{'='*56}")
    print(f"  You: {user_message}")
    print(f"{'='*56}")

    while True:
        response = agent.invoke(current_input, config=config)

        if not response.get("__interrupt__"):
            # LLM has all inputs → responded directly from its knowledge
            final = response["messages"][-1].content
            print(f"\n  Agent:\n")
            print(final)
            print(f"\n{'─'*56}\n")
            return response

        # Only one interrupt type: ask_user_input
        action_request = response["__interrupt__"][0].value["action_requests"][0]
        current_input  = handle_ask_user_input(action_request)


print("Loop ready.")

Loop ready.


---
## Demos

Same loop. Same agent. Same one tool. Different usecases — the LLM adapts.

In [8]:
# DEMO 1: Travel Planning
# LLM asks departure city, date, cabin class.
# Then responds with flight options from its own knowledge — no search_flights tool.

run_claude_style_agent(
    user_message="I am travelling from India to China can you find flights for me?",
    thread_id="travel-02"
)


  You: I am travelling from India to China can you find flights for me?

────────────────────────────────────────────────────────
  Which city in India are you flying from?

  1. Delhi (DEL)
  2. Mumbai (BOM)
  3. Bengaluru (BLR)
  4. Chennai (MAA)
  5. Hyderabad (HYD)
  6. Kolkata (CCU)
  7. Other
  (or type your own answer)
────────────────────────────────────────────────────────
  ✓ Hyderabad (HYD)

────────────────────────────────────────────────────────
  Which city in China are you flying to?

  1. Beijing (PEK)
  2. Shanghai (PVG)
  3. Guangzhou (CAN)
  4. Shenzhen (SZX)
  5. Chengdu (CTU)
  6. Other
  (or type your own answer)
────────────────────────────────────────────────────────
  ✓ Shenzhen (SZX)

────────────────────────────────────────────────────────
  What is your travel date?

  1. Today
  2. Tomorrow
  3. Within this week
  4. Within this month
  5. I'll specify a date
  (or type your own answer)
────────────────────────────────────────────────────────
  ✓ Today

  

{'messages': [HumanMessage(content='I am travelling from India to China can you find flights for me?', additional_kwargs={}, response_metadata={}, id='bd79ad9b-39cf-40a7-8f03-be039efc5860'),
  AIMessage(content=[{'id': 'toolu_014f8ho6MyL32aftUd8Vk38y', 'caller': {'type': 'direct'}, 'input': {'question': 'Which city in India are you flying from?', 'options': ['Delhi (DEL)', 'Mumbai (BOM)', 'Bengaluru (BLR)', 'Chennai (MAA)', 'Hyderabad (HYD)', 'Kolkata (CCU)', 'Other']}, 'name': 'ask_user_input', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cd1BZJFKXzdX7b5ipzAjw', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 1305, 'output_tokens': 129, 'output_tokens_details': None, 'server_tool_use'

In [9]:
# DEMO 2: Email Drafting
# LLM asks recipient name/context, tone.
# Then drafts the full email itself — no send_email tool.

run_claude_style_agent(
    user_message="Send a follow-up email to my client about the delayed project delivery",
    thread_id="email-01"
)


  You: Send a follow-up email to my client about the delayed project delivery

────────────────────────────────────────────────────────
  What is the name of your client (or company) you're writing to?

  1. I'll type the name below
  2. Skip / Use 'Dear Client'
  (or type your own answer)
────────────────────────────────────────────────────────
  ✓ Prudhvi

────────────────────────────────────────────────────────
  What tone should this follow-up email have?

  1. Apologetic and reassuring
  2. Professional and formal
  3. Friendly and empathetic
  4. Firm and solution-focused
  (or type your own answer)
────────────────────────────────────────────────────────
  ✓ Professional and formal

────────────────────────────────────────────────────────
  What is the reason for the project delay?

  1. Technical challenges / development issues
  2. Resource/team availability
  3. Dependency on third-party/vendor
  4. Scope changes requested by client
  5. I'll describe it below
  (or type you

{'messages': [HumanMessage(content='Send a follow-up email to my client about the delayed project delivery', additional_kwargs={}, response_metadata={}, id='7e7edd7e-3c2d-4526-940f-e9b4ace3fb0f'),
  AIMessage(content=[{'id': 'toolu_011FWcnF7heaBD5XYfHS2BCz', 'caller': {'type': 'direct'}, 'input': {'question': "What is the name of your client (or company) you're writing to?", 'options': ["I'll type the name below", "Skip / Use 'Dear Client'"]}, 'name': 'ask_user_input', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cd1CoLVGAdNK5rSKrzmKC', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 1305, 'output_tokens': 105, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'st

In [ ]:
# DEMO 3: Meeting Scheduling
# LLM asks attendees, duration, date preference.
# Then writes a full meeting invite with agenda — no schedule_meeting tool.

run_claude_style_agent(
    user_message="Schedule a sprint planning meeting for my engineering team",
    thread_id="meeting-01"
)

In [ ]:
# DEMO 4: Pharma Query — likely no clarification needed
# Drug name and context are clear in the message.
# LLM responds directly without calling ask_user_input.

run_claude_style_agent(
    user_message="What are the side effects of Metformin for a Type 2 diabetic patient?",
    thread_id="pharma-01"
)

In [ ]:
# DEMO 5: Content Writing
# LLM asks tone and angle, then writes the full post.

run_claude_style_agent(
    user_message="Write me a LinkedIn post about the future of GenAI agents",
    thread_id="linkedin-01"
)

In [ ]:
# DEMO 6: Multi-Turn — Same Thread
# InMemorySaver keeps the full conversation alive.
# Second message builds on the first without re-collecting inputs.

THREAD = "multiturn-01"

run_claude_style_agent(
    user_message="Help me plan a 3-day trip to Japan",
    thread_id=THREAD
)

run_claude_style_agent(
    user_message="Now suggest a hotel in Tokyo for that trip",
    thread_id=THREAD   # Same thread — agent remembers the Japan trip context
)

---
## Summary

### Architecture comparison

```
Claude (Anthropic):                    Our LangChain Agent:

  Tool: ask_user_input                   Tool: ask_user_input
  interrupt_on: {ask_user_input: True}   interrupt_on: {ask_user_input: True}
  Action engine: foundation model        Action engine: same LLM (its knowledge)
  Action tools: none                     Action tools: none
```

### What the LLM decides on its own

```
  WHEN to call ask_user_input   (what info is truly missing vs inferrable)
  WHAT question to ask          (specific to the request context)
  WHAT options to show          (contextually meaningful, not generic)
  HOW MANY times to ask         (as many as needed, one per missing piece)
  WHEN to stop asking           (once it has enough to respond completely)
  HOW to respond                (drafts, plans, info — all from its knowledge)
```

### The complete flow

```
agent.invoke({messages: [HumanMessage]}, config)
       │
       ├─ __interrupt__ absent?
       │    LLM had all context → responded directly
       │    Print response. Done.
       │
       └─ __interrupt__ present?
            LLM called ask_user_input → needs one more piece of info
            Show question + options
            Get human answer
            Command(resume={decisions:[{type:edit, edited_action:{args:{user_answer: ...}}}]})
            → loop
```

| | Notebook 3.3 | This notebook |
|---|---|---|
| **Tools** | `read_email`, `send_email` | `ask_user_input` only |
| **interrupt_on** | `{send_email: True}` | `{ask_user_input: True}` |
| **Action engine** | Tool calls real API | LLM responds from knowledge |
| **Adding new usecase** | Add a new tool | Nothing — LLM handles it |
| **Resume format** | `Command(resume={decisions:[...]})` | Same |